# Netflix Titles — Exploratory Data Analysis

**Week 2 Task: Exploratory Data Analysis**

This notebook picks up where Week 1 (`netflix_data_cleaning.ipynb`) left off. We re-run the same cleaning steps to rebuild the clean dataset, then explore it: descriptive stats, 5+ visualizations, and a written insight for each one.

**Dataset:** Netflix Movies and TV Shows (8,800+ rows) — [Kaggle source](https://www.kaggle.com/shivamb/netflix-shows) (CC0), via TidyTuesday mirror.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)


## 0. Rebuild the Cleaned Dataset

Same cleaning logic from Week 1 — repeated here so this notebook can run standalone. See `netflix_data_cleaning.ipynb` for the full explanation of each decision.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv"
df_raw = pd.read_csv(DATA_URL)

df = df_raw.drop_duplicates().copy()

text_cols = ['director', 'cast', 'country']
for col in text_cols:
    df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
    df[col] = df[col].fillna('Unknown')

df = df.dropna(subset=['date_added', 'rating']).copy()

df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

duration_num = df['duration'].str.extract(r'(\d+)').astype(float)
df['duration_minutes'] = np.where(df['type'] == 'Movie', duration_num[0], np.nan)
df['duration_seasons'] = np.where(df['type'] == 'TV Show', duration_num[0], np.nan)

print(f"Cleaned dataset ready: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 1. Descriptive Statistics

Before charting anything, let's get a numeric feel for the data: how content is split between movies and shows, and the shape of runtime/season-count distributions.


In [ ]:
print("Type breakdown:")
print(df['type'].value_counts())
print(f"\n{(df['type'].value_counts(normalize=True) * 100).round(1)}%")


In [ ]:
print("Movie duration (minutes) — descriptive stats:")
print(df['duration_minutes'].describe())

print("\nTV show length (seasons) — descriptive stats:")
print(df['duration_seasons'].describe())


In [ ]:
print("Release year — descriptive stats:")
print(df['release_year'].describe())

print("\nMost common content rating:")
print(df['rating'].value_counts().head(5))


**Quick reads from the numbers alone:**
- Movies mean duration (~99 min) sits close to the median, suggesting a fairly symmetric distribution with a long right tail (a handful of very long films).
- TV shows: the median season count is much lower than what a quick glance suggests — most shows only run 1 season, and a few long-running ones pull the mean up.
- Most content in the catalog was released fairly recently relative to the platform's history, which we'll see clearly once we chart it.


## 2. Visualizations

Each chart below is built to answer a specific question about the catalog, with the takeaway written immediately after.


### 2.1 — What's the mix of Movies vs. TV Shows?

In [ ]:
fig, ax = plt.subplots()
type_counts = df['type'].value_counts()
ax.bar(type_counts.index, type_counts.values, color=['#E50914', '#221f1f'])
ax.set_title('Netflix Catalog: Movies vs. TV Shows')
ax.set_xlabel('Content Type')
ax.set_ylabel('Number of Titles')
for i, v in enumerate(type_counts.values):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.show()


**Insight:** Movies make up roughly 69% of the catalog vs. 31% for TV shows — Netflix's library still skews heavily toward films, even though its original-series strategy gets a lot of the marketing attention.


### 2.2 — How long are Netflix movies, typically?

In [ ]:
fig, ax = plt.subplots()
ax.hist(df['duration_minutes'].dropna(), bins=30, color='#E50914', edgecolor='black')
ax.axvline(df['duration_minutes'].median(), color='black', linestyle='--', label=f"Median: {df['duration_minutes'].median():.0f} min")
ax.set_title('Distribution of Movie Durations')
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of Movies')
ax.legend()
plt.show()


**Insight:** Movie runtimes cluster tightly around 90–110 minutes (standard feature-length), with a right-skewed tail of longer films (documentaries, epics) stretching past 150 minutes. Very few movies run under an hour.


### 2.3 — How has content added to Netflix grown over time?

In [ ]:
yearly = df[df['year_added'] >= 2010].groupby(['year_added', 'type']).size().unstack(fill_value=0)

fig, ax = plt.subplots()
yearly.plot(ax=ax, marker='o', color=['#E50914', '#221f1f'])
ax.set_title('Titles Added to Netflix per Year, by Type')
ax.set_xlabel('Year Added')
ax.set_ylabel('Number of Titles Added')
ax.legend(title='Type')
plt.show()


**Insight:** Additions to the catalog ramped up sharply from 2015 onward and peaked around 2019, for both movies and TV shows — consistent with Netflix's aggressive content-acquisition phase before its growth slowed. The dip in the final year likely reflects a partial year in the dataset rather than an actual slowdown.


### 2.4 — Which countries produce the most titles on Netflix?

In [ ]:
top_countries = (
    df[df['country'] != 'Unknown']['country']
    .str.split(', ')
    .explode()
    .value_counts()
    .head(10)
)

fig, ax = plt.subplots()
ax.barh(top_countries.index[::-1], top_countries.values[::-1], color='#E50914')
ax.set_title('Top 10 Countries by Number of Titles')
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Country')
plt.show()


**Insight:** The United States dominates the catalog by a wide margin, with India a distant second — reflecting both Netflix's origins and its early push into the Bollywood-heavy Indian market. Note this undercounts multi-country co-productions' secondary countries slightly less than single-country attribution would.


### 2.5 — What are the most common genres?

In [ ]:
top_genres = (
    df['listed_in']
    .str.split(', ')
    .explode()
    .value_counts()
    .head(10)
)

fig, ax = plt.subplots()
sns.barplot(x=top_genres.values, y=top_genres.index, ax=ax, color='#221f1f')
ax.set_title('Top 10 Genres on Netflix')
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Genre')
plt.show()


**Insight:** International Movies and Dramas dominate the genre list, well ahead of niche categories like Documentaries or Anime — the catalog is built around broad, mainstream drama content rather than specialized genres.


### 2.6 — How is content rated (age/content rating)?

In [ ]:
fig, ax = plt.subplots()
rating_order = df['rating'].value_counts().index
sns.countplot(data=df, y='rating', order=rating_order, ax=ax, color='#E50914')
ax.set_title('Distribution of Content Ratings')
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Rating')
plt.show()


**Insight:** TV-MA and TV-14 together account for the majority of the catalog — Netflix's content skews toward mature/teen audiences rather than young children, which lines up with its adult-oriented original programming strategy.


## 3. Summary of Findings

| # | Question | Key Insight |
|---|---|---|
| 1 | Movie vs. TV Show mix | Catalog is ~69% movies, ~31% TV shows |
| 2 | Movie duration distribution | Clusters around 90–110 min, right-skewed tail of longer films |
| 3 | Growth over time | Sharp ramp-up in additions from 2015, peaking around 2019 |
| 4 | Top countries | US dominates, India a distant second |
| 5 | Top genres | International Movies and Dramas lead; broad mainstream content over niche genres |
| 6 | Content ratings | Skews toward mature audiences (TV-MA, TV-14) over children's content |

Every chart above was chosen to answer a specific question rather than to show off a plotting technique — each has a labeled axis, a title, and a one-line takeaway.
